# LiExNet-Improved v3 — Final Experiments: More Seeds + Wider Student

This notebook evaluates lightweight facial-expression-recognition students on FER2013, examining the effects of architectural modifications, knowledge distillation (KD), and increased student model capacity.

The experimental setup uses the same training recipe, data augmentation, model design approach, and knowledge-distillation configuration (temperature T=4, alpha=0.5) across the evaluated variants, with the student width configuration varied for the wider experiments.

### Experimental variants

1. **Narrow students (A, D, E):** Evaluated using seeds 0–4 (5 runs per variant).
2. **Wide students (A-wide, D-wide, E-wide):** Evaluated using `WIDTHS=(32,64,96,128)` and seeds 0–2 (3 runs per variant).

### Reproducibility and experiment recovery

- Random seeds are set before model construction to ensure corresponding variants begin with comparable initializations for paired comparisons.
- Completed runs and test probabilities are saved to Google Drive (`runs.csv` and probability files), allowing experiments to resume after interruptions.
- The teacher checkpoint is cached to avoid unnecessary retraining.
- The notebook generates paired statistical comparisons, seed-averaged per-class F1 scores, confusion matrices, and a consolidated master results table.

### Data split

The official FER2013 training partition (28,709 images) was used for model training. The official 7,178-image PrivateTest partition was treated as a held-out pool and split into validation and test subsets using stratified sampling (50/50, random seed 42). Therefore, the reported test metrics are based on a derived subset of the official PrivateTest partition, rather than the original unsplit PrivateTest set.

In [ ]:
import os, re, time, json, math, copy, random, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("Using device:", device, "| AMP:", USE_AMP)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed(42)

## 0. Configuration

In [ ]:
CFG = dict(
    IMG_SIZE=48, TEACHER_IMG=96, HEAD="gap",
    EPOCHS=60, BATCH=128, LR=2e-3, WD=5e-4, WARMUP_EPOCHS=3,
    LABEL_SMOOTH=0.1, EMA_DECAY=0.999, USE_CLASS_WEIGHTS=True,
    KD_T=4.0, KD_ALPHA=0.5,
    TEACHER_EPOCHS=30, TEACHER_LR=5e-4, TEACHER_WD=1e-4,
)

NARROW, WIDE = (24, 48, 72, 96), (32, 64, 96, 128)
SEEDS_NARROW, SEEDS_WIDE = [0, 1, 2, 3, 4], [0, 1, 2]

ARCH_BASE = dict(spatial=False, sk_blocks=(),   gate="legacy")     # LiExNet-style baseline (ECA only)
ARCH_IMP  = dict(spatial=True,  sk_blocks=(2,), gate="identity")   # + spatial gate + SK block, identity-init

VARIANTS = {
    "A_baseline":     dict(arch=ARCH_BASE, kd=False, widths=NARROW, seeds=SEEDS_NARROW),
    "D_improved":     dict(arch=ARCH_IMP,  kd=False, widths=NARROW, seeds=SEEDS_NARROW),
    "E_improved+KD":  dict(arch=ARCH_IMP,  kd=True,  widths=NARROW, seeds=SEEDS_NARROW),
    "A_wide":         dict(arch=ARCH_BASE, kd=False, widths=WIDE,   seeds=SEEDS_WIDE),
    "D_wide":         dict(arch=ARCH_IMP,  kd=False, widths=WIDE,   seeds=SEEDS_WIDE),
    "E_wide+KD":      dict(arch=ARCH_IMP,  kd=True,  widths=WIDE,   seeds=SEEDS_WIDE),
}
print("Total student runs:", sum(len(v["seeds"]) for v in VARIANTS.values()))

## 1. Load FER2013 and preload it on the GPU

In [ ]:
# --- Locate the CSV (Drive search; falls back to local Colab files) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

FILE_PATH = None
for cand in ("/content/FER.csv", "/content/FER__1_.csv", "FER.csv", "fer2013.csv"):
    if os.path.exists(cand):
        FILE_PATH = cand; break
if FILE_PATH is None and os.path.isdir("/content/drive/MyDrive"):
    for root, dirs, filenames in os.walk("/content/drive/MyDrive"):
        if "FER.csv" in filenames:
            FILE_PATH = os.path.join(root, "FER.csv"); break
assert FILE_PATH is not None, "FER csv not found: check the filename / that the Drive mount succeeded"
print("Dataset:", FILE_PATH, f"({os.path.getsize(FILE_PATH) / 2**20:.1f} MB)")

# --- Output folder (Drive if available, so runs survive a disconnect) ---
OUT_DIR = "/content/drive/MyDrive/liexnet_v3" if os.path.isdir("/content/drive/MyDrive") else "/content/liexnet_v3"
PROBS_DIR = os.path.join(OUT_DIR, "probs")
os.makedirs(PROBS_DIR, exist_ok=True)
RUNS_CSV = os.path.join(OUT_DIR, "runs.csv")
TEACHER_PATH = os.path.join(OUT_DIR, "teacher_resnet18.pt")
print("Outputs will be saved to:", OUT_DIR)

In [ ]:
def load_fer_table(path):
    df = pd.read_csv(path)
    cols = {c.strip().lower(): c for c in df.columns}
    for name in ("emotion", "pixels", "usage"):
        if name not in cols:
            raise ValueError(f"Column '{name}' not found. Available: {list(df.columns)}")
    return df, cols["emotion"], cols["pixels"], cols["usage"]

df, EMOTION_COL, PIXELS_COL, USAGE_COL = load_fer_table(FILE_PATH)
print(df.shape)
print(df[USAGE_COL].astype(str).str.strip().value_counts())

EMOTION_NAMES = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
NUM_CLASSES = len(EMOTION_NAMES)

usage = df[USAGE_COL].astype(str).str.strip().str.lower()
is_train = usage.isin(["training", "train"])
is_pub   = usage.isin(["publictest", "public_test", "public test", "val", "validation"])
is_priv  = usage.isin(["privatetest", "private_test", "private test", "test"])

train_df = df[is_train].reset_index(drop=True)
if is_pub.any() and is_priv.any():
    val_df, test_df = df[is_pub].reset_index(drop=True), df[is_priv].reset_index(drop=True)
    print("Using the official Training / PublicTest / PrivateTest split.")
else:
    pool = df[is_pub | is_priv]
    val_df, test_df = train_test_split(pool, test_size=0.5, stratify=pool[EMOTION_COL], random_state=42)
    val_df, test_df = val_df.reset_index(drop=True), test_df.reset_index(drop=True)
    print("NOTE: no PublicTest label in this CSV -> held-out pool split 50/50 (stratified, seed 42) into val/test.")
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

def to_tensors(d):
    X = np.stack([np.array(p.split(), dtype=np.uint8) for p in d[PIXELS_COL].values])
    assert X.shape[1] == 48 * 48, f"Unexpected pixel count {X.shape[1]} (expected 2304)"
    X = torch.from_numpy(X.reshape(-1, 1, 48, 48)).to(device)
    y = torch.from_numpy(d[EMOTION_COL].values.astype(np.int64)).to(device)
    return X, y

Xtr, ytr = to_tensors(train_df); Xva, yva = to_tensors(val_df); Xte, yte = to_tensors(test_df)

counts = torch.bincount(ytr, minlength=NUM_CLASSES).float()
CLASS_W = ((counts.sum() / (NUM_CLASSES * counts)) ** 0.5)
CLASS_W = CLASS_W / CLASS_W.mean()
print("Train class counts :", counts.long().tolist())
print("Class weights      :", [round(v, 2) for v in CLASS_W.tolist()])

## 2. Preprocessing and GPU augmentation (unchanged from v2)

In [ ]:
_xf = Xtr.float() / 255.0
STU_MEAN, STU_STD = _xf.mean().item(), _xf.std().item()
del _xf
IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IM_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def student_prep(x):
    s = CFG["IMG_SIZE"]
    if x.shape[-1] != s:
        x = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False)
    return (x - STU_MEAN) / STU_STD

def teacher_prep(x):
    t = CFG["TEACHER_IMG"]
    x = F.interpolate(x.repeat(1, 3, 1, 1), size=(t, t), mode="bilinear", align_corners=False)
    return (x - IM_MEAN) / IM_STD

def random_erase(x, p=0.25, area=(0.02, 0.15)):
    B, _, H, W = x.shape; dev = x.device
    apply = torch.rand(B, device=dev) < p
    a = torch.empty(B, device=dev).uniform_(*area) * H * W
    r = torch.empty(B, device=dev).uniform_(math.log(0.3), math.log(3.3)).exp()
    h = torch.sqrt(a * r).clamp(1, H); w = torch.sqrt(a / r).clamp(1, W)
    top = torch.rand(B, device=dev) * (H - h); left = torch.rand(B, device=dev) * (W - w)
    ys = torch.arange(H, device=dev).view(1, H, 1).float()
    xs = torch.arange(W, device=dev).view(1, 1, W).float()
    m = ((ys >= top.view(B, 1, 1)) & (ys < (top + h).view(B, 1, 1)) &
         (xs >= left.view(B, 1, 1)) & (xs < (left + w).view(B, 1, 1)) & apply.view(B, 1, 1))
    fill = torch.rand(B, 1, 1, 1, device=dev).expand_as(x)
    return torch.where(m.unsqueeze(1), fill, x)

def gpu_augment(x):
    B = x.size(0); dev = x.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(10)
    sc  = 1 + (torch.rand(B, device=dev) * 2 - 1) * 0.10
    tx  = (torch.rand(B, device=dev) * 2 - 1) * 0.10
    ty  = (torch.rand(B, device=dev) * 2 - 1) * 0.10
    flip = (torch.rand(B, device=dev) < 0.5).float() * 2 - 1
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos * flip, -sin, tx], 1),
                         torch.stack([sin * flip,  cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
    c = 1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.2
    b = (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.1
    x = ((x - 0.5) * c + 0.5 + b).clamp(0, 1)
    return random_erase(x)

def iterate(X, y, bs, shuffle=False, drop_last=False):
    n = len(X)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    end = n - (n % bs) if drop_last else n
    for i in range(0, end, bs):
        j = idx[i:i + bs]
        yield X[j], y[j]

## 3. Models (architecture code unchanged; `build_student` now takes the widths)

In [ ]:
class ECA(nn.Module):
    def __init__(self, ch, gamma=2, b=1, gate="identity"):
        super().__init__()
        k = int(abs(math.log2(ch) / gamma + b)); k = k if k % 2 else k + 1
        self.gate = gate
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
        if gate == "identity":
            nn.init.zeros_(self.conv.weight)
    def forward(self, x):
        y = self.conv(x.mean((2, 3)).unsqueeze(1)).squeeze(1)
        s = torch.sigmoid(y)
        if self.gate == "identity":
            s = 2 * s
        return x * s.view(s.size(0), s.size(1), 1, 1)

class SpatialGate(nn.Module):
    def __init__(self, ch, k=5, gate="identity"):
        super().__init__()
        self.gate = gate
        self.dw = nn.Conv2d(ch, ch, k, padding=k // 2, groups=ch, bias=True)
        if gate == "identity":
            nn.init.zeros_(self.dw.weight); nn.init.zeros_(self.dw.bias)
    def forward(self, x):
        s = torch.sigmoid(self.dw(x))
        if self.gate == "identity":
            s = 2 * s
        return x * s

class Attention(nn.Module):
    def __init__(self, ch, spatial, gate):
        super().__init__()
        self.eca = ECA(ch, gate=gate)
        self.sp = SpatialGate(ch, 5, gate) if spatial else nn.Identity()
    def forward(self, x):
        return self.sp(self.eca(x))

class SKLite(nn.Module):
    def __init__(self, ch, reduction=8):
        super().__init__()
        self.dw3 = nn.Conv2d(ch, ch, 3, padding=1, groups=ch, bias=False)
        self.dw5 = nn.Conv2d(ch, ch, 5, padding=2, groups=ch, bias=False)
        hidden = max(ch // reduction, 8)
        self.fc1 = nn.Linear(ch, hidden); self.fc2 = nn.Linear(hidden, 2 * ch)
        self.bn = nn.BatchNorm2d(ch)
    def forward(self, x):
        f3, f5 = self.dw3(x), self.dw5(x)
        h = (f3 + f5).mean((2, 3))
        w = F.softmax(self.fc2(F.relu(self.fc1(h))).view(-1, 2, x.size(1)), dim=1)
        a = w[:, 0].unsqueeze(-1).unsqueeze(-1); b = w[:, 1].unsqueeze(-1).unsqueeze(-1)
        return self.bn(F.relu(a * f3 + b * f5))

class FeatureBlock(nn.Module):
    def __init__(self, in_ch, out_ch, use_sk, spatial, gate):
        super().__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True), nn.BatchNorm2d(out_ch))
        self.dw = SKLite(out_ch) if use_sk else nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, groups=out_ch), nn.ReLU(inplace=True), nn.BatchNorm2d(out_ch))
        self.attn = Attention(out_ch, spatial, gate)
        self.conv2 = nn.Sequential(nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2))
    def forward(self, x):
        return self.conv2(self.attn(self.dw(self.conv1(x))))

class LiExNetV(nn.Module):
    def __init__(self, widths=(24, 48, 72, 96), spatial=False, sk_blocks=(), gate="identity",
                 head="gap", img=48, in_ch=1, num_classes=NUM_CLASSES):
        super().__init__()
        chs = [in_ch] + list(widths)
        self.blocks = nn.Sequential(*[FeatureBlock(chs[i], chs[i + 1], i in sk_blocks, spatial, gate)
                                      for i in range(len(widths))])
        if head == "gap":
            pool, feat = nn.AdaptiveAvgPool2d(1), widths[-1]
        else:
            pool, feat = nn.Identity(), widths[-1] * (img // 2 ** len(widths)) ** 2
        self.head = nn.Sequential(pool, nn.Flatten(), nn.Linear(feat, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.head(self.blocks(x))

def build_student(arch, widths):
    return LiExNetV(widths=widths, head=CFG["HEAD"], img=CFG["IMG_SIZE"], in_ch=1, **arch).to(device)

def build_teacher():
    import torchvision
    try:
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        print("Teacher: ImageNet-pretrained ResNet-18")
    except Exception as e:
        print("Could not load pretrained weights (", type(e).__name__, ") -> teacher trained from scratch")
        m = torchvision.models.resnet18(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, NUM_CLASSES))
    return m.to(device)

for name, v in VARIANTS.items():
    m = build_student(v["arch"], v["widths"])
    print(f"{name:16s} widths={v['widths']}  params = {sum(p.numel() for p in m.parameters()):>8,}")

## 4. Training utilities (unchanged from v2)

In [ ]:
class EMA:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.step = decay, 0
    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = min(self.decay, (1 + self.step) / (10 + self.step))
        for e, m in zip(self.module.state_dict().values(), model.state_dict().values()):
            if e.dtype.is_floating_point:
                e.mul_(d).add_(m.detach(), alpha=1 - d)
            else:
                e.copy_(m)

@torch.no_grad()
def predict_probs(model, X, prep, tta=False, bs=512):
    model.eval(); outs = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].float() / 255.0
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            p = F.softmax(model(prep(x)).float(), 1)
            if tta:
                p = (p + F.softmax(model(prep(torch.flip(x, dims=[3]))).float(), 1)) / 2
        outs.append(p)
    return torch.cat(outs)

def score(probs, y):
    pred, yt = probs.argmax(1).cpu().numpy(), y.cpu().numpy()
    return accuracy_score(yt, pred), f1_score(yt, pred, average="macro")

def _make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_model(model, prep, tag, epochs, lr, wd, teacher=None, kd_T=4.0, kd_alpha=0.5,
                class_w=None, seed=0, log_every=10):
    set_seed(seed)
    model = model.to(device)
    ema = EMA(model, CFG["EMA_DECAY"])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    spe = len(Xtr) // CFG["BATCH"]
    total, warm = epochs * spe, CFG["WARMUP_EPOCHS"] * spe
    def lr_lambda(s):
        if s < warm:
            return (s + 1) / warm
        p = (s - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * p))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    scaler = _make_scaler()
    best_acc, best_state, best_ep = -1.0, None, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in iterate(Xtr, ytr, CFG["BATCH"], shuffle=True, drop_last=True):
            x = gpu_augment(xb.float() / 255.0)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(prep(x)).float()
                loss = F.cross_entropy(logits, yb, weight=class_w, label_smoothing=CFG["LABEL_SMOOTH"])
                if teacher is not None:
                    with torch.no_grad():
                        tl = teacher(teacher_prep(x)).float()
                    kd = F.kl_div(F.log_softmax(logits / kd_T, 1), F.softmax(tl / kd_T, 1),
                                  reduction="batchmean") * kd_T * kd_T
                    loss = (1 - kd_alpha) * loss + kd_alpha * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model)
        val_acc, _ = score(predict_probs(ema.module, Xva, prep), yva)
        if val_acc > best_acc:
            best_acc, best_ep = val_acc, ep + 1
            best_state = {k: v.detach().clone() for k, v in ema.module.state_dict().items()}
        if (ep + 1) % log_every == 0 or ep == 0 or ep + 1 == epochs:
            print(f"[{tag}] ep {ep+1:3d}/{epochs}  loss={loss.item():.3f}  val_acc(EMA)={val_acc:.4f}  "
                  f"best={best_acc:.4f}@{best_ep}  ({time.time()-t0:.0f}s)")
    ema.module.load_state_dict(best_state)
    return ema.module, best_acc, best_ep

## 5. Teacher (trained once, cached to Drive so every student sees the same teacher)

In [ ]:
teacher = build_teacher()
if os.path.exists(TEACHER_PATH):
    teacher.load_state_dict(torch.load(TEACHER_PATH, map_location=device))
    print("Loaded cached teacher from", TEACHER_PATH)
else:
    teacher, t_val, t_ep = train_model(teacher, teacher_prep, "teacher", CFG["TEACHER_EPOCHS"],
                                       CFG["TEACHER_LR"], CFG["TEACHER_WD"],
                                       class_w=CLASS_W if CFG["USE_CLASS_WEIGHTS"] else None, seed=0, log_every=5)
    torch.save(teacher.state_dict(), TEACHER_PATH)
    print("Saved teacher to", TEACHER_PATH)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
t_acc, t_f1 = score(predict_probs(teacher, Xte, teacher_prep), yte)
t_acc_tta, _ = score(predict_probs(teacher, Xte, teacher_prep, tta=True), yte)
print(f"\nTeacher: test={t_acc:.4f}  test+TTA={t_acc_tta:.4f}  macroF1={t_f1:.4f}")
TEACHER_TEST_ACC, TEACHER_TEST_ACC_TTA = t_acc, t_acc_tta

## 6. Experiments: all variants x all seeds (resumable)
Each finished run is appended to `runs.csv` on Drive, and its flip-TTA test probabilities are saved. If the session dies, rerun all cells: finished runs are skipped.

In [ ]:
def prob_path(vname, seed):
    return os.path.join(PROBS_DIR, re.sub(r"[^A-Za-z0-9]+", "_", vname) + f"__s{seed}.npy")

rows = pd.read_csv(RUNS_CSV).to_dict("records") if os.path.exists(RUNS_CSV) else []
done = {(r["variant"], int(r["seed"])) for r in rows}
class_w = CLASS_W if CFG["USE_CLASS_WEIGHTS"] else None
print(f"{len(done)} runs already finished.")

for vname, v in VARIANTS.items():
    for seed in v["seeds"]:
        if (vname, seed) in done:
            print(f"skip {vname} seed {seed} (already in runs.csv)")
            continue
        set_seed(seed)                                   # seed BEFORE building -> paired initialisation
        model = build_student(v["arch"], v["widths"])
        n_params = sum(p.numel() for p in model.parameters())
        model, val_acc, best_ep = train_model(
            model, student_prep, f"{vname} s{seed}", CFG["EPOCHS"], CFG["LR"], CFG["WD"],
            teacher=teacher if v["kd"] else None, kd_T=CFG["KD_T"], kd_alpha=CFG["KD_ALPHA"],
            class_w=class_w, seed=seed, log_every=20)
        p_plain = predict_probs(model, Xte, student_prep)
        p_tta = predict_probs(model, Xte, student_prep, tta=True)
        acc, f1 = score(p_plain, yte); acc_tta, f1_tta = score(p_tta, yte)
        rows.append(dict(variant=vname, seed=seed, params=n_params, val_acc=val_acc, best_epoch=best_ep,
                         test_acc=acc, test_acc_tta=acc_tta, macro_f1=f1, macro_f1_tta=f1_tta))
        done.add((vname, seed))
        pd.DataFrame(rows).to_csv(RUNS_CSV, index=False)
        np.save(prob_path(vname, seed), p_tta.cpu().numpy().astype(np.float16))
        print(f"  -> {vname} seed {seed}: test={acc:.4f}  test+TTA={acc_tta:.4f}  macroF1={f1:.4f}  [saved]\n")

res = pd.DataFrame(rows)
res["variant"] = pd.Categorical(res["variant"], categories=list(VARIANTS), ordered=True)
res = res.sort_values(["variant", "seed"]).reset_index(drop=True)
print("All runs finished:", len(res))

## 7. Summary (mean +/- std over seeds)

In [ ]:
g = res.groupby("variant", observed=True)
def ms(col):
    return g[col].mean().mul(100).round(2).astype(str) + " +/- " + g[col].std(ddof=1).fillna(0).mul(100).round(2).astype(str)

summary = pd.DataFrame({
    "n seeds": g.size(), "params": g["params"].first(),
    "val acc %": ms("val_acc"), "test acc %": ms("test_acc"),
    "test acc % (TTA)": ms("test_acc_tta"), "macro F1 %": ms("macro_f1"), "macro F1 % (TTA)": ms("macro_f1_tta"),
})
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 40)
print(summary.to_string())
print(f"\nTeacher test acc: {100*TEACHER_TEST_ACC:.2f}%   (with flip TTA: {100*TEACHER_TEST_ACC_TTA:.2f}%)")

## 8. Paired comparisons (same seed, difference in percentage points)
`mean diff` is the average per-seed difference; `95% CI` uses a t-interval, `p` is a paired t-test. With 3 to 5 seeds these are weak tests and the p-values are not corrected for multiple comparisons: treat them as a guide, not proof. A difference whose CI includes 0 should be reported as "no detectable difference".

In [ ]:
from scipy import stats

PAIRS = [
    ("A_baseline",    "D_improved",    "architecture effect, narrow (D - A)"),
    ("D_improved",    "E_improved+KD", "KD effect, narrow (E - D)"),
    ("A_baseline",    "E_improved+KD", "total, narrow (E - A)"),
    ("A_wide",        "D_wide",        "architecture effect, wide (D - A)"),
    ("D_wide",        "E_wide+KD",     "KD effect, wide (E - D)"),
    ("A_wide",        "E_wide+KD",     "total, wide (E - A)"),
    ("A_baseline",    "A_wide",        "capacity effect, baseline (wide - narrow)"),
    ("E_improved+KD", "E_wide+KD",     "capacity effect, improved+KD (wide - narrow)"),
]
def paired_table(metric):
    out = []
    for a, b, label in PAIRS:
        pa = res[res.variant == a].set_index("seed")[metric]
        pb = res[res.variant == b].set_index("seed")[metric]
        common = pa.index.intersection(pb.index)
        if len(common) < 2:
            continue
        d = (pb[common] - pa[common]).values * 100
        n = len(d); sd = d.std(ddof=1); se = sd / np.sqrt(n)
        ci = stats.t.ppf(0.975, n - 1) * se
        p = stats.ttest_1samp(d, 0.0).pvalue if sd > 0 else float("nan")
        out.append({"comparison": label, "n": n, "mean diff (pp)": round(d.mean(), 2),
                    "95% CI": f"[{d.mean()-ci:+.2f}, {d.mean()+ci:+.2f}]", "p": round(p, 3),
                    "CI excludes 0": bool(d.mean() - ci > 0 or d.mean() + ci < 0)})
    return pd.DataFrame(out)

for metric in ("test_acc", "test_acc_tta", "macro_f1"):
    print(f"\n=== {metric} ===")
    print(paired_table(metric).to_string(index=False))

## 9. Per-class F1 (seed-averaged) and confusion matrices (seed-summed)

In [ ]:
import matplotlib.pyplot as plt
gt = yte.cpu().numpy()

pc_rows, cms = [], {v: np.zeros((NUM_CLASSES, NUM_CLASSES)) for v in VARIANTS}
for r in res.itertuples():
    pred = np.load(prob_path(r.variant, r.seed)).astype(np.float32).argmax(1)
    f1s = f1_score(gt, pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    pc_rows.append(dict(variant=r.variant, **{n: f for n, f in zip(EMOTION_NAMES, f1s)}))
    cms[r.variant] += confusion_matrix(gt, pred, labels=list(range(NUM_CLASSES)))

per_class = pd.DataFrame(pc_rows).groupby("variant", observed=True).mean().mul(100).round(1)
print("Per-class F1 % (flip TTA, mean over seeds):")
print(per_class.to_string())

names = [v for v in VARIANTS if cms[v].sum() > 0]
ncol = 3; nrow = math.ceil(len(names) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, name in zip(axes.ravel(), names):
    ax.axis("on")
    cmn = cms[name] / cms[name].sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(EMOTION_NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(EMOTION_NAMES, fontsize=8)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_title(name, fontsize=10); ax.set_xlabel("Predicted", fontsize=8); ax.set_ylabel("True", fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "confusion_matrices.png"), dpi=200); plt.show()

e = cms["E_improved+KD"]
print("E_improved+KD (summed over seeds): Sad->Neutral =", int(e[4, 6]), "| Neutral->Sad =", int(e[6, 4]))

## 10. Efficiency (Colab proxies, NOT a Jetson benchmark)
Cost does not depend on trained weights or on KD, so fresh models are used. FLOPs count conv/linear ops only. Latency: single CPU thread, 50 warm-up + 300 timed runs, median.

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def efficiency(model):
    m = copy.deepcopy(model).cpu().eval()
    S = CFG["IMG_SIZE"]; x = torch.randn(1, 1, S, S)
    try:
        with FlopCounterMode(display=False) as fc:
            m(x)
        gflops = fc.get_total_flops() / 1e9
    except Exception:
        gflops = float("nan")
    with tempfile.NamedTemporaryFile(suffix=".pt") as f:
        torch.save(m.state_dict(), f.name); size_mb = os.path.getsize(f.name) / 2**20
    old = torch.get_num_threads(); torch.set_num_threads(1)
    with torch.no_grad():
        for _ in range(50): m(x)
        ts = []
        for _ in range(300):
            t = time.perf_counter(); m(x); ts.append((time.perf_counter() - t) * 1000)
    torch.set_num_threads(old)
    return dict(GFLOPs=round(gflops, 4), size_MB=round(size_mb, 3), latency_ms_median=round(float(np.median(ts)), 3))

eff = pd.DataFrame({name: efficiency(build_student(v["arch"], v["widths"])) for name, v in VARIANTS.items()}).T
print(eff.to_string())
print("\nBase paper (Jetson TX2): params=42,000  GFLOPs=0.086  size=0.17 MB  latency=1.88 ms")

## 11. Master table and export

In [ ]:
master = summary.join(eff)
print(master.to_string())

master.to_csv(os.path.join(OUT_DIR, "master_table.csv"))
per_class.to_csv(os.path.join(OUT_DIR, "per_class_f1.csv"))
paired_table("test_acc").to_csv(os.path.join(OUT_DIR, "paired_test_acc.csv"), index=False)
paired_table("macro_f1").to_csv(os.path.join(OUT_DIR, "paired_macro_f1.csv"), index=False)
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump({"CFG": CFG, "NARROW": NARROW, "WIDE": WIDE, "SEEDS_NARROW": SEEDS_NARROW, "SEEDS_WIDE": SEEDS_WIDE,
               "teacher_test_acc": TEACHER_TEST_ACC, "teacher_test_acc_tta": TEACHER_TEST_ACC_TTA}, f, indent=2)
print("\nAll files saved in:", OUT_DIR)